# Motor de Reglas Heurísticas — Caso 3

**Input:** `data/casos_con_features.parquet` (150 casos con las 16 features del step 02).

**Objetivo:** evaluar cada caso y emitir **APROBAR**, **RECHAZAR** o **ESCALAR**.

**Flujo del pipeline:** ``datos base → FeatureEngineer → RuleEngine → decisión``

Las reglas están implementadas en ``src/rules/rule_engine.py`` y los thresholds en ``src/rules/thresholds.yaml``. La política completa se explica en ``docs/politicas_decision.md``.

## 1. Setup

Cargamos el dataset con features y creamos el RuleEngine.

In [1]:
import sys
from pathlib import Path

import pandas as pd

# Asegurar que el proyecto esté en el path
PROYECTO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROYECTO) not in sys.path:
    sys.path.insert(0, str(PROYECTO))

from src.rules.rule_engine import RuleEngine

DATA_DIR = Path('data') if Path('data').exists() else Path('../data')
df = pd.read_parquet(DATA_DIR / 'casos_con_features.parquet')

engine = RuleEngine()

print(f'Casos cargados: {len(df)}')
print(f'Reglas cargadas desde: {engine.t["rechazar"]["flags_minimos"]["fuente"]}')

Casos cargados: 150
Reglas cargadas desde: EDA: flags_fraude_previos correlaciona r=0.74 con num_compensaciones_90d


## 2. Aplicar reglas

Ejecutamos ``engine.decide(df)`` que aplica todas las reglas en orden de precedencia:

1. ESCALAR forzoso (palabras críticas)
2. RECHAZAR (cualquier regla de fraude)
3. APROBAR (todas las condiciones de legitimidad)
4. Resto → ESCALAR por ambigüedad

In [2]:
resultado = engine.decide(df)

print('Distribución de decisiones:')
print(resultado['recomendacion'].value_counts().to_string())
print(f'\nPorcentajes:')
print((resultado['recomendacion'].value_counts(normalize=True) * 100).round(1).astype(str) + '%')

Distribución de decisiones:
recomendacion
ESCALAR     92
RECHAZAR    49
APROBAR      9

Porcentajes:
recomendacion
ESCALAR     61.3%
RECHAZAR    32.7%
APROBAR      6.0%
Name: proportion, dtype: str


### Lectura

El gráfico anterior muestra cuántos casos caen en cada bucket. El objetivo es que la mayoría de los casos **no fraudulentos y no ambiguos** se automaticen (APROBAR o RECHAZAR), y que solo los casos genuinamente ambiguos queden en ESCALAR (revisión humana).

## 3. Ejemplos por categoría

Mostramos 2 ejemplos representativos de cada decisión para validar que el motor está clasificando correctamente.

In [3]:
print('--- RECHAZAR (señales de fraude) ---')
mask_rej = resultado['recomendacion'] == 'RECHAZAR'
if mask_rej.any():
    for _, r in resultado[mask_rej].head(2).iterrows():
        print(f'  Caso {r["caso_id"]}: {r["justificacion"][:200]}')
        print(f'  Señales: {r["senales_usadas"]}')


--- RECHAZAR (señales de fraude) ---
  Caso COMP-0006: RECHAZAR por señales de fraude: compensacion > p99 (604.64).
  Señales: compensacion > p99 (604.64)
  Caso COMP-0009: RECHAZAR por señales de fraude: flags_fraude_previos >= 2.
  Señales: flags_fraude_previos >= 2


In [4]:
print('--- APROBAR (casos legítimos) ---')
mask_apr = resultado['recomendacion'] == 'APROBAR'
if mask_apr.any():
    for _, r in resultado[mask_apr].head(2).iterrows():
        print(f'  Caso {r["caso_id"]}: {r["justificacion"][:200]}')
        print(f'  Señales: {r["senales_usadas"]}')


--- APROBAR (casos legítimos) ---
  Caso COMP-0012: APROBAR: caso consistente (gps_ok_sano).
  Señales: gps_ok_sano
  Caso COMP-0032: APROBAR: caso consistente (gps_ok_sano).
  Señales: gps_ok_sano


In [5]:
print('--- ESCALAR (ambigüedad / palabras críticas) ---')
mask_esc = resultado['recomendacion'] == 'ESCALAR'
if mask_esc.any():
    for _, r in resultado[mask_esc].head(2).iterrows():
        print(f'  Caso {r["caso_id"]}: {r["justificacion"][:200]}')
        print(f'  Señales: {r["senales_usadas"]}')


--- ESCALAR (ambigüedad / palabras críticas) ---
  Caso COMP-0001: ESCALAR por ambigüedad: las reglas no encuentran señales claras ni de fraude ni de legitimidad. Requiere análisis del texto del reclamo por LLM.
  Señales: ambiguo: requiere análisis LLM
  Caso COMP-0002: ESCALAR por seguridad de marca: el reclamo contiene palabras críticas que requieren revisión humana.
  Señales: flag_palabras_criticas: seguridad de marca


## 4. Validación cruzada

Verificamos que el motor produce los tres tipos de decisión y que no hay valores nulos en las columnas de salida.

In [6]:
print('Validación de integridad:')
print(f'  Total casos: {len(resultado)}')
print(f'  NaN en recomendacion: {int(resultado["recomendacion"].isna().sum())}')
print(f'  NaN en senales_usadas: {int(resultado["senales_usadas"].isna().sum())}')
print(f'  NaN en justificacion:  {int(resultado["justificacion"].isna().sum())}')
print(f'  Categorías: {resultado["recomendacion"].unique().tolist()}')

assert not resultado['recomendacion'].isna().any(), 'NaN en recomendacion'
print('\n[OK] Validación superada.')

Validación de integridad:
  Total casos: 150
  NaN en recomendacion: 0
  NaN en senales_usadas: 0
  NaN en justificacion:  0
  Categorías: ['ESCALAR', 'RECHAZAR', 'APROBAR']

[OK] Validación superada.


## 5. Guardado de resultados

Persistimos las decisiones para que el pipeline LangGraph (step 06) pueda leerlas. Se guarda como parquet y se actualiza PostgreSQL.

In [7]:
OUTPUT_PARQUET = DATA_DIR / 'casos_con_reglas.parquet'
resultado.to_parquet(OUTPUT_PARQUET, index=False)
print(f'[OK] Parquet: {OUTPUT_PARQUET}')

[OK] Parquet: ../data/casos_con_reglas.parquet


In [ ]:
import psycopg2

DB_CONFIG = {
    'host': 'localhost', 'port': 5432, 'dbname': 'rappi_cases',
    'user': 'rappi', 'password': 'rappi_pass',
}

conn = psycopg2.connect(**DB_CONFIG)
actualizados = 0
try:
    with conn.cursor() as cur:
        for _, r in resultado.iterrows():
            cur.execute(
                '''INSERT INTO resolution_case
                       (caso_id, fuente, decision, decision_regla,
                        justificacion_regla, senales_regla)
                   VALUES (%s, %s, %s, %s, %s, %s)
                   ON CONFLICT (caso_id) DO UPDATE SET
                       fuente = EXCLUDED.fuente,
                       decision = EXCLUDED.decision,
                       decision_regla = EXCLUDED.decision_regla,
                       justificacion_regla = EXCLUDED.justificacion_regla,
                       senales_regla = EXCLUDED.senales_regla,
                       updated_at = NOW()''',
                (r['caso_id'], 'reglas', r['recomendacion'], r['recomendacion'],
                 r['justificacion'], r['senales_usadas']),
            )
            actualizados += cur.rowcount
    conn.commit()
finally:
    conn.close()

print(f'[OK] resolution_case actualizado: {actualizados} casos')

In [ ]:
# Verificación final
conn = psycopg2.connect(**DB_CONFIG)
check = pd.read_sql(
    '''SELECT decision, COUNT(*) AS n
       FROM resolution_case
       WHERE caso_id IN (SELECT caso_id FROM casos WHERE es_sintetico = FALSE)
       GROUP BY decision
       ORDER BY n DESC''',
    conn,
)
conn.close()
print(check.to_string(index=False))
print('\n[OK] Step 03 completo.')